In [1]:
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings
from dotenv import load_dotenv
import os

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))
api_key = os.getenv("key") 

chat_model = create_chat_model(api_key=api_key, model="gpt-4o-mini", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

car_template = """You are an expert in automobiles. You have extensive knowledge about car mechanics, \
models, and automotive technology. You provide clear and helpful answers about cars.

Here is a question:
{query}"""

restaurant_template = """You are a knowledgeable food critic and restaurant reviewer. You have a deep understanding of \
different cuisines, dining experiences, and what makes a great restaurant. You answer questions about restaurants insightfully.

Here is a question:
{query}"""

technology_template = """You are a tech expert with in-depth knowledge of the latest gadgets, software, \
and technological trends. You provide insightful and detailed answers about technology.

Here is a question:
{query}"""

c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
car_questions = [
    "What is the difference between a sedan and an SUV?",
    "How does a hybrid car save fuel?",
    "What should I look for when buying a used car?",
]

restaurant_questions = [
    "What makes a five-star restaurant exceptional?",
    "How do I choose a good wine pairing for my meal?",
    "What are the key elements of French cuisine?",
]

technology_questions = [
    "What are the latest advancements in AI?",
    "How do I secure my home network against cyber threats?",
    "What should I consider when buying a new smartphone?",
]

In [3]:
embeddings = embeddings

car_question_embeddings = embeddings.embed_documents(car_questions)
restaurant_question_embeddings = embeddings.embed_documents(restaurant_questions)
technology_question_embeddings = embeddings.embed_documents(technology_questions)

In [4]:
def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    car_similarity = cosine_similarity([query_embedding], car_question_embeddings)[0]
    restaurant_similarity = cosine_similarity(
        [query_embedding], restaurant_question_embeddings
    )[0]
    technology_similarity = cosine_similarity(
        [query_embedding], technology_question_embeddings
    )[0]

    max_similarity = max(
        max(car_similarity), max(restaurant_similarity), max(technology_similarity)
    )

    if max_similarity == max(car_similarity):
        print("Using CAR")
        return PromptTemplate.from_template(car_template)
    elif max_similarity == max(restaurant_similarity):
        print("Using RESTAURANT")
        return PromptTemplate.from_template(restaurant_template)
    else:
        print("Using TECHNOLOGY")
        return PromptTemplate.from_template(technology_template)


input_query = {"query": "What's the best way to improve my cars's battery life?"}
prompt = prompt_router(input_query)

Using CAR


In [5]:
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | chat_model
    | StrOutputParser()
)

In [6]:
chain.invoke("How do I identify a good vintage wine at a restaurant?")

Using RESTAURANT


"Identifying a good vintage wine at a restaurant involves a combination of knowledge, observation, and sometimes, intuition. Here are some key points to consider:\n\n1. **Understanding Vintage**: Familiarize yourself with the concept of vintage wines. A vintage wine is made from grapes harvested in a specific year, and the quality can vary greatly based on the climate and conditions of that year. Research which vintages are highly regarded for specific regions and varietals.\n\n2. **Wine List Review**: When examining the wine list, look for a well-curated selection. A restaurant that takes pride in its wine offerings will often have a diverse range of vintages, regions, and styles. Pay attention to wines from reputable producers, as they are more likely to have high-quality offerings.\n\n3. **Storage and Condition**: Good restaurants will store their wines properly, away from light, at controlled temperatures, and with bottles laid down. If you have the opportunity, observe the wine ce

Classification

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

classification_template = PromptTemplate.from_template(
    """You are good at classifying a question.
    Given the user question below, classify it as either being about `Car`, `Restaurant`, or `Technology`.

    <If the question is about car mechanics, models, or automotive technology, classify it as 'Car'>
    <If the question is about cuisines, dining experiences, or restaurant services, classify it as 'Restaurant'>
    <If the question is about gadgets, software, or technological trends, classify it as 'Technology'>

    <question>
    {question}
    </question>

    Classification:"""
)

classification_chain = classification_template | chat_model | StrOutputParser()

In [9]:
def prompt_router(input):
    classification = classification_chain.invoke({"question": input["query"]})

    if classification == "Car":
        print("Using CAR")
        return PromptTemplate.from_template(car_template)
    elif classification == "Restaurant":
        print("Using RESTAURANT")
        return PromptTemplate.from_template(restaurant_template)
    elif classification == "Technology":
        print("Using TECHNOLOGY")
        return PromptTemplate.from_template(technology_template)
    else:
        print("Unexpected classification:", classification)
        return None


input_query = {"query": "What are the latest trends in electric cars?"}
prompt = prompt_router(input_query)

Using CAR


In [12]:
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | chat_model
    | StrOutputParser()
)

In [13]:
chain.invoke("How do I identify a good vintage wine at a restaurant?")

Using RESTAURANT


"Identifying a good vintage wine at a restaurant involves a combination of understanding wine labels, the restaurant's wine program, and some sensory evaluation. Here are several key points to consider:\n\n1. **Understand the Vintage**: The vintage year on a wine bottle indicates the year the grapes were harvested. Some years are renowned for producing exceptional wines due to favorable weather conditions. Research the region and grape varietals to understand which vintages are considered outstanding. For instance, a 2015 Bordeaux is often celebrated, while some years may be less favorable.\n\n2. **Check the Wine List**: A well-curated wine list often reflects the restaurant's commitment to quality. Look for descriptions that highlight the wine's origin, producer, and tasting notes. If the list features a diverse selection from various regions and vintages, it suggests a knowledgeable sommelier or wine buyer.\n\n3. **Ask the Sommelier**: Don’t hesitate to engage with the sommelier or w